# Reibungsereignisse am Tragband: Ortung mit einem einzelnen Sensor
### Dispersion als Lineal, amortisierte Bayes-Inversion, selbstgeplante Folgemessung

Beim Ablassen einer Detektorkette an einem dünnen Tragband stellt ein einzelner Beschleunigungssensor an der Winde zunächst nur fest, *dass* es irgendwo reibt. Operativ entscheidend ist aber *wo*: Kontakt an der Schleusenöffnung ist eine andere Situation als ein Verkanten nahe der Nutzlast. Ortung gilt klassisch als Mehrsensor-Problem (Triangulation über Laufzeitdifferenzen).

Ein dünnes Band bietet jedoch einen physikalischen Ausweg: Biegewellen sind **dispersiv** ($\omega = a k^2$, Gruppengeschwindigkeit $v_g = 2\sqrt{a\omega}$) – hohe Frequenzen laufen schneller. Ein Stoß in Tiefe $d$ kommt oben als **Chirp** an, dessen Zeit-Frequenz-Struktur die Laufstrecke kodiert. Die Entfernung steckt damit in *einer einzigen* Zeitreihe. Dieselbe Idee trägt die Guided-Wave-Ortung im Structural Health Monitoring.

**Der Haken, der das Problem interessant macht:** Die Dispersionskonstante $a$ hängt von der Bandspannung ab und ist im Betrieb nicht genau bekannt (Nutzlast, Temperatur, Trommellage). Der Chirp misst nur die Kombination $d/\sqrt{a}$ – Tiefe und Spannung sind aus dem passiven Signal **entartet**. Behandelt wird das hier als Inversionsproblem mit drei Bausteinen:

1. **Amortisierte neuronale Posterior-Schätzung** $p(d, s, a \mid x)$ aus simulierten Signalen – einmal trainiert, danach Inferenz in Millisekunden; die Entartung erscheint als explizite Korrelationsstruktur statt als Fit-Instabilität.
2. **Kalibrierungsnachweis (SBC):** Ein Posterior ist nur so gut wie seine Unsicherheiten; die simulationsbasierte Kalibrierung prüft sie quantitativ.
3. **Bayessche Versuchsplanung:** Der erwartete Informationsgewinn vergleicht zwei Handlungsoptionen – auf ein weiteres passives Ereignis warten oder einen **aktiven Ping** senden, dessen Echo über die bekannte Bandlänge $2L$ die Spannung direkt misst und die Entartung bricht.

Alles läuft auf CPU (Training gesamt einige Minuten). Benötigt `torch` und `scipy` (in Colab vorhanden).

In [ ]:
import numpy as np, torch, torch.nn as nn, time
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.stats import kstest
torch.manual_seed(0); np.random.seed(0)

# ---------------- Vorwärtsmodell ----------------
FS = 20000.0; N = 4096
L_BAND = 8.0                                   # aufgehängte Bandlänge [m], bekannt
D_MIN, D_MAX = 0.5, 7.5                        # Ereignistiefe d [m]
S_MIN, S_MAX = 0.3, 2.0                        # Stoßstärke s [a.u.]
A_MIN, A_MAX = 15.0, 40.0                      # Dispersionskonstante a [m²/s] ~ Bandspannung
ETA, NOISE = 4e-6, 0.02
f_r = np.fft.rfftfreq(N, 1/FS); om = 2*np.pi*f_r
SRC = 20.0*np.exp(-((f_r-800)/900)**2)*(f_r > 20)      # Spektrum eines kurzen Stoßes

def passiv_batch(D, S, A, rng):
    """Stoß der Stärke s in Tiefe d; dispergiert+gedämpft am Sensor (Synthese im f-Raum)."""
    D = np.asarray(D)[:,None]; S = np.asarray(S)[:,None]; A = np.asarray(A)[:,None]
    k = np.sqrt(np.maximum(om,1e-9)[None]/A)                    # omega = a k²
    H = np.exp(-1j*k*D)*np.exp(-ETA*om[None]*D)
    return (np.fft.irfft(S*SRC[None]*H, n=N, axis=1) + rng.normal(0,NOISE,(len(D),N))).astype(np.float32)

def ping_batch(A, rng, s_ping=1.0):
    """Aktiver Ping an der Winde; Echo vom unteren Bandende: Laufweg 2·L_BAND, bekannt."""
    A = np.asarray(A)[:,None]
    k = np.sqrt(np.maximum(om,1e-9)[None]/A)
    H = np.exp(-1j*k*(2*L_BAND))*np.exp(-ETA*om[None]*(2*L_BAND))
    return (np.fft.irfft(s_ping*SRC[None]*H, n=N, axis=1) + rng.normal(0,NOISE,(len(A),N))).astype(np.float32)

t = np.arange(N)/FS
fig, ax = plt.subplots(1, 2, figsize=(11,3.6))
for d in [1.5, 4.0, 7.0]:
    ax[0].plot(t*1000, passiv_batch([d],[1.2],[20.0], np.random.default_rng(1))[0], lw=.7, label=f'd = {d} m')
ax[0].set_xlim(0,50); ax[0].legend(); ax[0].set_xlabel('t [ms]'); ax[0].grid(alpha=.3)
ax[0].set_title('Dispersion als Lineal: tieferes Ereignis = längerer Chirp')
for a_ in [15.0, 40.0]:
    ax[1].plot(t*1000, ping_batch([a_], np.random.default_rng(2))[0], lw=.7, label=f'a = {a_}')
ax[1].set_xlim(0,80); ax[1].legend(); ax[1].set_xlabel('t [ms]'); ax[1].grid(alpha=.3)
ax[1].set_title('Aktiver Ping: die Echo-Laufzeit über 2L misst die Spannung')
plt.tight_layout(); plt.show()

## Posterior-Schätzer: drei bewusste Konstruktionsentscheidungen

Eingabe ist ein grobes log-Spektrogramm (16 Frequenzbänder × 32 Zeitfenster) – die Chirp-Signatur lebt genau in dieser Zeit-Frequenz-Ebene, und die Mittelung macht die Darstellung robust gegen Phasenlage und Rauschrealisierung. Der Dichteschätzer ist ein Mixture-Density-Netz mit drei Eigenschaften, die hier keine Kür sind, sondern jeweils ein konkretes Versagen beheben (ein naiv trainierter Standard-MDN besteht die spätere Kalibrierungsprüfung nicht):

1. **Vollkovarianz-Komponenten** (Cholesky-parametrisiert): Der erwartete Posterior ist eine dünne, schräge Rippe entlang $d \propto \sqrt{a}$. Achsenparallele (diagonale) Gauß-Komponenten können sich nicht schräg legen – der Fit wird wahlweise zu breit oder falsch platziert.
2. **Logit-Transformation der Parameter:** Bei beschränktem Prior häufen sich Kalibrierungsfehler am Rand des Trägers (wahre Werte nahe der Grenze landen systematisch in den Extremrängen). Die Dichte wird deshalb im unbeschränkten Logit-Raum gelernt und zurücktransformiert.
3. **Early Stopping + affiner Debias:** Ausgewählt wird der Zustand mit bester Validierungs-NLL (Training bis zum Ende macht den Posterior überkonfident); ein danach auf frischen Simulationen gemessener mittlerer Restshift (~0.2σ) wird subtrahiert.

In [ ]:
def tf_features(X, chunk=1000):
    outs = []
    for c in range(0, len(X), chunk):
        _,_,Z = stft(X[c:c+chunk], FS, nperseg=256, noverlap=128, axis=-1)
        P = np.abs(Z)**2
        fb = np.array_split(np.arange(P.shape[-2]),16); tb = np.array_split(np.arange(P.shape[-1]),32)
        o = np.zeros((len(P),16,32), np.float32)
        for i,fi in enumerate(fb):
            for j,tj in enumerate(tb): o[:,i,j] = P[:,fi][:,:,tj].mean(axis=(1,2))
        outs.append(np.log10(o+1e-12).reshape(len(P),-1))
    return np.concatenate(outs)

class FCMDN(nn.Module):
    """Mixture-Density-Netz mit Vollkovarianz je Komponente (Cholesky)."""
    def __init__(self, d_in, d_out=3, K=8, hid=128):
        super().__init__(); self.K, self.d = K, d_out
        self.register_buffer('tril_ix', torch.tril_indices(d_out,d_out,-1))
        self.body = nn.Sequential(nn.Linear(d_in,hid), nn.ReLU(), nn.Linear(hid,hid), nn.ReLU())
        self.pi = nn.Linear(hid,K); self.mu = nn.Linear(hid,K*d_out)
        self.ld = nn.Linear(hid,K*d_out); self.lo = nn.Linear(hid,K*(d_out*(d_out-1)//2))
    def params(self, x):
        h = self.body(x); B = x.shape[0]
        logpi = torch.log_softmax(self.pi(h),1)
        mu = self.mu(h).view(B,self.K,self.d)
        Ld = self.ld(h).view(B,self.K,self.d).clamp(-6,3)
        Lo = self.lo(h).view(B,self.K,-1)
        L = torch.zeros(B,self.K,self.d,self.d)
        L[:,:,torch.arange(self.d),torch.arange(self.d)] = Ld.exp()
        L[:,:,self.tril_ix[0],self.tril_ix[1]] = Lo
        return logpi, mu, L, Ld
    def nll(self, x, y):
        logpi, mu, L, Ld = self.params(x)
        r = (y.unsqueeze(1)-mu).unsqueeze(-1)
        z = torch.linalg.solve_triangular(L, r, upper=False).squeeze(-1)
        comp = -0.5*(z**2).sum(-1) - Ld.sum(-1) - 0.5*self.d*np.log(2*np.pi)
        return -(torch.logsumexp(logpi+comp,1)).mean()
    def sample(self, x, n=400):
        with torch.no_grad():
            logpi, mu, L, _ = self.params(x)
            k = torch.multinomial(logpi.exp().expand(n,-1),1).squeeze(1)
            eps = torch.randn(n,self.d,1)
            return (mu[0,k] + (L[0,k]@eps).squeeze(-1)).numpy()

LO_ = np.array([D_MIN,S_MIN,A_MIN]); HI_ = np.array([D_MAX,S_MAX,A_MAX])
nth = lambda th: ((th-LO_)/(HI_-LO_)).astype(np.float32)
dn  = lambda z: z*(HI_-LO_)+LO_
EPSL = 1e-3
to_logit   = lambda Y: np.log(np.clip(Y,EPSL,1-EPSL)/(1-np.clip(Y,EPSL,1-EPSL))).astype(np.float32)
from_logit = lambda Z: 1/(1+np.exp(-Z))

def train_logit(F, Y, ep=150, val_frac=0.1, seed=0):
    torch.manual_seed(seed)
    Yl = to_logit(Y); nval = int(len(F)*val_frac)
    mx, sx = F[:-nval].mean(0), F[:-nval].std(0)+1e-6
    my, sy = Yl[:-nval].mean(0), Yl[:-nval].std(0)+1e-6
    Xt, Yt = torch.tensor((F[:-nval]-mx)/sx), torch.tensor((Yl[:-nval]-my)/sy)
    Xv, Yv = torch.tensor((F[-nval:]-mx)/sx), torch.tensor((Yl[-nval:]-my)/sy)
    net = FCMDN(F.shape[1]); opt = torch.optim.Adam(net.parameters(), 1.5e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, ep)
    best, bs = 1e9, None
    for e in range(ep):
        perm = torch.randperm(len(Xt))
        for b in range(0, len(Xt), 1024):
            i = perm[b:b+1024]; opt.zero_grad(); l = net.nll(Xt[i],Yt[i]); l.backward(); opt.step()
        sch.step()
        with torch.no_grad(): lv = float(net.nll(Xv,Yv))
        if lv < best: best, bs = lv, {k:v.clone() for k,v in net.state_dict().items()}
    net.load_state_dict(bs)
    return net, mx, sx, my, sy

def post_raw(net, mx, sx, my, sy, f, n=3000):
    z = net.sample(torch.tensor(((f-mx)/sx)[None]), n)
    return dn(from_logit(z*sy+my))

In [ ]:
# ---------------- NPE_1: passives Signal -> (d, s, a) ----------------
NSIM = 20000
rng = np.random.default_rng(1); t0 = time.time()
D = rng.uniform(D_MIN,D_MAX,NSIM); S = rng.uniform(S_MIN,S_MAX,NSIM); A = rng.uniform(A_MIN,A_MAX,NSIM)
F1 = np.concatenate([tf_features(passiv_batch(D[c:c+2000],S[c:c+2000],A[c:c+2000],rng),2000) for c in range(0,NSIM,2000)])
Y = nth(np.stack([D,S,A],1))
net1, mx1, sx1, my1, sy1 = train_logit(F1, Y)

# affiner Debias auf frischen Simulationen
rngC = np.random.default_rng(77); dl = []
for _ in range(200):
    dd = rngC.uniform(D_MIN,D_MAX); ss = rngC.uniform(S_MIN,S_MAX); aa = rngC.uniform(A_MIN,A_MAX)
    f = tf_features(passiv_batch([dd],[ss],[aa],rngC)[:1])[0]
    dl.append(post_raw(net1,mx1,sx1,my1,sy1,f,200).mean(0) - np.array([dd,ss,aa]))
shift1 = np.array(dl).mean(0)

def post1(f, n=3000):
    return np.clip(post_raw(net1,mx1,sx1,my1,sy1,f,n) - shift1, LO_, HI_)

print(f'NPE_1 trainiert und kalibriert in {time.time()-t0:.0f}s  (Debias-Shift: {np.round(shift1,3)})')

## Kalibrierungsnachweis (SBC)

Zieht man Parameter aus dem Prior, simuliert eine Messung und bestimmt den Rang des wahren Werts unter den Posterior-Stichproben, müssen die Ränge bei korrekter Inferenz gleichverteilt sein – jede Abweichung verrät Über-/Unterkonfidenz oder Bias. Erst dieser Test macht die späteren Unsicherheitsangaben belastbar.

In [ ]:
rngS = np.random.default_rng(33); ranks = []
for _ in range(300):
    dd = rngS.uniform(D_MIN,D_MAX); ss = rngS.uniform(S_MIN,S_MAX); aa = rngS.uniform(A_MIN,A_MAX)
    f = tf_features(passiv_batch([dd],[ss],[aa],rngS)[:1])[0]
    ps = post1(f, 400)
    ranks.append([(ps[:,0]<dd).mean(), (ps[:,1]<ss).mean(), (ps[:,2]<aa).mean()])
ranks = np.array(ranks)
fig, ax = plt.subplots(1, 3, figsize=(11,3))
for j, nm in enumerate(['Tiefe d','Stärke s','Spannung a']):
    p = kstest(ranks[:,j],'uniform').pvalue
    ax[j].hist(ranks[:,j], bins=12, color='steelblue', alpha=.85, edgecolor='w')
    ax[j].axhline(len(ranks)/12, color='crimson', ls='--')
    ax[j].set_title(f'{nm}   (KS p = {p:.2f})'); ax[j].set_xlabel('SBC-Rang')
plt.suptitle('Ränge ~ gleichverteilt  ⇒  die Posterior-Unsicherheiten sind kalibriert')
plt.tight_layout(); plt.show()

## Die Tiefe–Spannungs-Entartung, offen ausgesprochen

Drei Ereignisse bei identischer Bandkonfiguration: flach, tief, mittel. Der Posterior in der $(a,d)$-Ebene zeigt jeweils die Rippe entlang $d \propto \sqrt{a}$ – und die Invariante $d/\sqrt{a}$ ist um ein Mehrfaches schärfer bestimmt als $d$ selbst. Das ist die quantitative Fassung von: *Der Chirp misst die Laufzeit, nicht die Strecke.*

In [ ]:
szenarien = [(1.5,1.0,20.0,'flaches Ereignis'), (6.5,1.0,20.0,'tiefes Ereignis'), (4.0,1.0,20.0,'mittleres Ereignis')]
fig, ax = plt.subplots(1, 3, figsize=(13,4))
fobs = None
for a_,(d_t,s_t,a_t,titel) in zip(ax, szenarien):
    f = tf_features(passiv_batch([d_t],[s_t],[a_t], np.random.default_rng(7))[:1])[0]
    ps = post1(f)
    if titel.startswith('mittleres'): fobs, p1s = f, ps      # Fall für die Versuchsplanung
    inv = ps[:,0]/np.sqrt(ps[:,2])
    a_.scatter(ps[:,2], ps[:,0], s=4, alpha=.12, color='slateblue')
    a_.scatter([a_t],[d_t], color='crimson', marker='x', s=140, label='wahr')
    a_.set_xlim(A_MIN,A_MAX); a_.set_ylim(D_MIN,D_MAX)
    a_.set_xlabel('a (Bandspannung)'); a_.grid(alpha=.3); a_.legend(loc='lower right')
    a_.set_title(f'{titel}\nσ(d)={ps[:,0].std():.2f} m,  σ(d/√a)={inv.std():.3f}  (r={np.corrcoef(ps[:,0],ps[:,2])[0,1]:+.2f})')
ax[0].set_ylabel('Tiefe d [m]')
plt.suptitle('Passiv allein: der Posterior legt die Entartungsrichtung offen, statt eine Scheingenauigkeit zu liefern')
plt.tight_layout(); plt.show()
print(f'Mittleres Ereignis:  d = {p1s[:,0].mean():.2f} ± {p1s[:,0].std():.2f} m   |   d/√a = {(p1s[:,0]/np.sqrt(p1s[:,2])).mean():.3f} ± {(p1s[:,0]/np.sqrt(p1s[:,2])).std():.3f}')

## Versuchsplanung: passiv warten oder aktiv pingen?

Zwei Handlungsoptionen, formal verglichen über den erwarteten Informationsgewinn $\mathrm{EIG} = H[p(\theta\mid x_1)] - \mathbb{E}\,H[p(\theta\mid x_1, x_2)]$, wobei die möglichen Ausgänge $x_2$ aus dem aktuellen Posterior simuliert werden – die wahren Parameter werden dafür nicht benötigt:

- **Option A – warten:** ein weiteres passives Ereignis (gleiche Quelle, neue Rausch-Realisierung). Erwartung: wenig Gewinn, denn es misst wieder nur $d/\sqrt{a}$.
- **Option B – aktiver Ping:** bekannte Anregung an der Winde, Echo über $2L$. Erwartung: großer Gewinn, denn die Laufzeit über eine *bekannte* Strecke misst $a$ isoliert – die zur Rippe senkrechte Information.

Für beide Optionen wird ein eigener amortisierter Schätzer trainiert, der jeweils beide Beobachtungen gemeinsam verarbeitet.

In [ ]:
t0 = time.time()
F2a = np.concatenate([tf_features(passiv_batch(D[c:c+2000],S[c:c+2000],A[c:c+2000],rng),2000) for c in range(0,NSIM,2000)])
net2p, mx2p, sx2p, my2p, sy2p = train_logit(np.concatenate([F1,F2a],1), Y, seed=1)
Fg  = np.concatenate([tf_features(ping_batch(A[c:c+2000],rng),2000) for c in range(0,NSIM,2000)])
net2g, mx2g, sx2g, my2g, sy2g = train_logit(np.concatenate([F1,Fg],1), Y, seed=2)

# Debias für den Ping-Schätzer (liefert am Ende die quantitative Aussage)
rngC = np.random.default_rng(88); dl = []
for _ in range(150):
    dd = rngC.uniform(D_MIN,D_MAX); ss = rngC.uniform(S_MIN,S_MAX); aa = rngC.uniform(A_MIN,A_MAX)
    fp = tf_features(passiv_batch([dd],[ss],[aa],rngC)[:1])[0]
    fq = tf_features(ping_batch([aa],rngC)[:1])[0]
    dl.append(post_raw(net2g,mx2g,sx2g,my2g,sy2g,np.concatenate([fp,fq]),200).mean(0) - np.array([dd,ss,aa]))
shift2g = np.array(dl).mean(0)
def post2g(f, n=3000):
    return np.clip(post_raw(net2g,mx2g,sx2g,my2g,sy2g,f,n) - shift2g, LO_, HI_)
def post2p(f, n=3000):
    return post_raw(net2p,mx2p,sx2p,my2p,sy2p,f,n)
print(f'Beide Zwei-Beobachtungs-Schätzer trainiert in {time.time()-t0:.0f}s')

def entropie(ps):
    c = np.cov(ps.T) + 1e-9*np.eye(3)
    return 0.5*np.log(np.linalg.det(2*np.pi*np.e*c))

H1 = entropie(p1s); rngE = np.random.default_rng(11)
hs_p, hs_g = [], []
for dd,ss,aa in p1s[rngE.integers(0,len(p1s),30)]:
    f2 = tf_features(passiv_batch([dd],[ss],[aa],rngE)[:1])[0]
    hs_p.append(entropie(post2p(np.concatenate([fobs,f2]),1500)))
    fq = tf_features(ping_batch([aa],rngE)[:1])[0]
    hs_g.append(entropie(post2g(np.concatenate([fobs,fq]),1500)))
eig_p = (H1-np.mean(hs_p))/np.log(2); eig_g = (H1-np.mean(hs_g))/np.log(2)
print(f'EIG  Option A (weiteres passives Ereignis): {eig_p:5.1f} bit')
print(f'EIG  Option B (aktiver Ping):               {eig_g:5.1f} bit   <- Empfehlung')

In [ ]:
# Die empfohlene Messung ausführen (wahres a der Anlage: 20.0)
d_t, s_t, a_t = 4.0, 1.0, 20.0
fq = tf_features(ping_batch([a_t], np.random.default_rng(8))[:1])[0]
p2s = post2g(np.concatenate([fobs,fq]))

fig, ax = plt.subplots(1, 2, figsize=(12,4.4))
ax[0].bar(['weiteres passives\nEreignis','aktiver Ping'], [eig_p,eig_g], color=['indianred','seagreen'])
ax[0].set_ylabel('erwarteter Informationsgewinn [bit]'); ax[0].grid(alpha=.3, axis='y')
ax[0].set_title('Der Algorithmus bewertet die Optionen – und empfiehlt den Ping')
ax[1].scatter(p1s[:,2], p1s[:,0], s=4, alpha=.10, color='indianred', label='nur passiv')
ax[1].scatter(p2s[:,2], p2s[:,0], s=4, alpha=.30, color='seagreen', label='passiv + Ping')
ax[1].scatter([a_t],[d_t], color='black', marker='x', s=150, label='wahr')
ax[1].set_xlim(A_MIN,A_MAX); ax[1].set_ylim(D_MIN,D_MAX)
ax[1].set_xlabel('a (Bandspannung)'); ax[1].set_ylabel('Tiefe d [m]'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('Der Ping liefert die zur Rippe senkrechte Information')
plt.tight_layout(); plt.show()

print(f'nur passiv:    d = {p1s[:,0].mean():.2f} ± {p1s[:,0].std():.2f} m')
print(f'passiv + Ping: d = {p2s[:,0].mean():.2f} ± {p2s[:,0].std():.2f} m   (wahr {d_t} m)')
print(f'realisierter Informationsgewinn: {(H1-entropie(p2s))/np.log(2):.1f} bit')

## Automatisierte Tests

In [ ]:
p_sbc = [kstest(ranks[:,j],'uniform').pvalue for j in range(3)]
assert min(p_sbc) > 0.01, f'SBC-Kalibrierung verletzt: {p_sbc}'
assert np.corrcoef(p1s[:,0], p1s[:,2])[0,1] > 0.6, 'd-a-Entartung sollte als Korrelation sichtbar sein'
assert (p1s[:,0]/np.sqrt(p1s[:,2])).std() < 0.5*p1s[:,0].std(), 'Invariante d/sqrt(a) sollte deutlich schärfer sein als d'
assert eig_g > eig_p + 1.0, 'Ping sollte klar mehr Information versprechen als Warten'
assert p2s[:,0].std() < 0.5*p1s[:,0].std(), 'Ping sollte die d-Unsicherheit deutlich reduzieren'
assert abs(p2s[:,0].mean()-4.0) < 3*p2s[:,0].std()+0.15, 'wahre Tiefe sollte im Posterior liegen'
print('✅ Alle Tests bestanden: kalibriert, Entartung sichtbar, Ping empfohlen und wirksam.')

---
### Einordnung und Grenzen

Das Wellenmodell ist bewusst minimal: eine Euler-Bernoulli-Biegemode mit $\omega = ak^2$, spannungs- und ortskonstantes $a$, frequenzproportionale Pauschal-Dämpfung, feste Quellsignatur, keine Reflexionen an Zwischenstrukturen oder am Ereignisort selbst. Ein reales Band bringt mehrere Moden, eine spannungsabhängige Dispersion mit Saiten-Anteil ($\omega^2 = (T/\mu)k^2 + (EI/\mu)k^4$), Temperaturgradienten entlang des Weges ins flüssige Argon und variable Stick-Slip-Spektren mit. All das gehört ins Vorwärtsmodell – der Rest der Kette (Posterior-Schätzung, Kalibrierung, Versuchsplanung) bleibt unverändert und erbt jede Physik, die man zu simulieren bereit ist. Die Quellsignatur ist die kritischste Vereinfachung: In einer ernsthaften Fassung würde ihre Form mit in den Parameterraum aufgenommen und mitmarginalisiert.

Erwähnenswert ist, was auf dem Weg hierher schiefging, weil es typisch ist: Ein naiv bis zum Ende trainierter Dichteschätzer *sah* überzeugend aus und war dennoch überkonfident – erst die SBC-Prüfung deckte es auf, und erst Vollkovarianz, Logit-Transformation und Debias zusammen bestanden sie. Ein Posterior ohne Kalibrierungsnachweis ist ein Bild, keine Messung.

**Methodische Referenzen:** Cranmer, Brehmer, Louppe, *The frontier of simulation-based inference* (PNAS 2020); Papamakarios & Murray (NeurIPS 2016) zur neuronalen Posterior-Schätzung; Talts et al. (2018) zur simulationsbasierten Kalibrierung; Lindley (1956), Ryan et al. (2016) zur bayesschen Versuchsplanung; zur Guided-Wave-Ortung mit dispersiven Wellen z. B. Croxford et al., *Strategies for guided-wave structural health monitoring* (Proc. R. Soc. A 2007).